# English–Sanskrit Neural Machine Translation using NLLB-200

#### Name: Aishwarya Mahindru
#### Roll No.: G25AIT1012

This notebook builds an English–Sanskrit neural machine translation system by fine-tuning Meta's NLLB-200 model on the provided parallel corpus. Translation quality is evaluated using BLEU and BERTScore.

## How to Run the Notebook

1. Open `G25AIT1012.ipynb` in Google Colab.
2. Configure the dataset paths and hyperparameters in the `Config` class.
3. Run all notebook cells.

The notebook performs the complete workflow:
- Downloads and prepares the dataset.
- Fine-tunes the NLLB-200 model.
- Evaluates the model on the validation set.
- Generates predictions and saves them as `submission.csv`.
- Saves the fine-tuned model as `finetuned_model.zip`.

### Running on a New Test Dataset

To generate predictions for a different test dataset using this same workflow, we need to update the test dataset path in the `Config` class (i.e., `private_test_path`) and run the notebook again. The notebook will automatically produce a new `submission.csv` for the specified dataset.

### 1. Installing Required Libraries

In [1]:
%%capture
!pip -q install \
transformers \
datasets \
evaluate \
accelerate \
sentencepiece \
sacrebleu \
sacremoses \
bert-score \
gdown \
nltk \
pandas \
numpy
!pip install -U bert-score
print("Dependencies Installed")

### 2. Importing Libraries

In [2]:
import inspect
import os
import time
import random
import shutil

import numpy as np
import pandas as pd

import torch

import nltk
nltk.download("punkt", quiet=True)

from dataclasses import dataclass

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

from nltk.translate.bleu_score import (
    corpus_bleu,
    sentence_bleu,
    SmoothingFunction,
)

### 3. Setting Random Seed

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

### 4. Configure Hyperparameters

In [4]:
from dataclasses import dataclass

@dataclass
class Config:

    # Execution
    train_mode: bool = True

    # Model
    model_name = "facebook/nllb-200-distilled-600M"
    src_lang = "san_Deva"
    tgt_lang = "eng_Latn"

    # Dataset
    data_dir = "data"

    train_src = "train_sa_10000.csv"
    train_tgt = "train_en_10000.csv"

    dev_src = "dev_sa_1000.csv"
    dev_tgt = "dev_en_1000.csv"

    test_src = "test_sa_1000.csv"
    test_tgt = "test_en_1000.csv"

    id_col = "Source_id"
    src_text_col = "Sentence_sa"
    tgt_text_col = "Sentence_en"

    # Inference
    model_dir = "finetuned_model"
    model_url = ""

    private_test_path = "data/test_sa_1000.csv"
    private_id_col = "Source_id"
    private_src_col = "Sentence_sa"

    # Sequence lengths
    max_len = 128
    max_decode_len = 256

    # Training
    epochs: int = 10
    lr: float = 2e-5

    batch_size = 8
    grad_accum = 2

    weight_decay = 0.01
    warmup_ratio = 0.10
    label_smoothing = 0.10

    early_stop_patience = 2

    # Decoding
    beam_size = 6

    # Output
    work_dir = "hf_out"

    # Reproducibility
    seed = 42


cfg = Config()

### 5. Prepare the Runtime

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")

if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

Device : cuda
NVIDIA A100-SXM4-40GB


### 6. Helper Functions

The following helper functions are used for loading the dataset, generating translations, and evaluating the model.

In [6]:
def load_pairs(cfg, src_file, tgt_file=None, data_dir=None):
    """Loading the source and target datasets."""

    data_dir = data_dir or cfg.data_dir

    src = pd.read_csv(os.path.join(data_dir, src_file))
    src[cfg.src_text_col] = src[cfg.src_text_col].fillna("").astype(str)

    if tgt_file is None:
        return (
            src[cfg.id_col].tolist(),
            src[cfg.src_text_col].tolist(),
            None,
        )

    tgt = pd.read_csv(os.path.join(data_dir, tgt_file))
    tgt[cfg.tgt_text_col] = tgt[cfg.tgt_text_col].fillna("").astype(str)

    merged = src.merge(tgt, on=cfg.id_col, how="inner")
    merged = merged[
        (merged[cfg.src_text_col].str.strip() != "")
        & (merged[cfg.tgt_text_col].str.strip() != "")
    ]

    return (
        merged[cfg.id_col].tolist(),
        merged[cfg.src_text_col].tolist(),
        merged[cfg.tgt_text_col].tolist(),
    )


@torch.no_grad()
def translate_corpus(model, tokenizer, texts, cfg, batch_size=32):
    """Generating translations for a list of source sentences."""

    model.eval()

    previous_cache = model.config.use_cache
    model.config.use_cache = True

    tokenizer.src_lang = cfg.src_lang
    forced_bos = tokenizer.convert_tokens_to_ids(cfg.tgt_lang)

    translations = []

    for i in range(0, len(texts), batch_size):

        batch = [
            text if text.strip() else " "
            for text in texts[i : i + batch_size]
        ]

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=cfg.max_len,
        ).to(model.device)

        generated = model.generate(
            **encoded,
            forced_bos_token_id=forced_bos,
            max_length=cfg.max_decode_len,
            num_beams=cfg.beam_size,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

        translations.extend(
            tokenizer.batch_decode(
                generated,
                skip_special_tokens=True,
            )
        )

    model.config.use_cache = previous_cache

    return translations


def compute_bleu(hypotheses, references):
    """Computing corpus-level and sentence-level BLEU scores."""

    tokenized_hypotheses = [hyp.split() for hyp in hypotheses]
    tokenized_references = [[ref.split()] for ref in references]

    corpus = corpus_bleu(
        tokenized_references,
        tokenized_hypotheses,
    )

    smoothing = SmoothingFunction().method1

    sentence = float(
        np.mean(
            [
                sentence_bleu(
                    ref,
                    hyp,
                    smoothing_function=smoothing,
                )
                for hyp, ref in zip(
                    tokenized_hypotheses,
                    tokenized_references,
                )
            ]
        )
    )

    return corpus, sentence


def compute_bertscore(hypotheses, references, lang="en"):
    """Computing the average BERTScore F1."""

    from bert_score import score

    hypotheses = [
        sentence if sentence.strip() else "<EMPTY>"
        for sentence in hypotheses
    ]

    _, _, f1 = score(
        hypotheses,
        references,
        lang=lang,
        rescale_with_baseline=True,
        use_fast_tokenizer=True,
        verbose=False,
    )

    return float(f1.mean())


class DataCollatorWithDecoderInputs(DataCollatorForSeq2Seq):
    """Custom data collator for sequence-to-sequence training."""

    def __call__(self, features, return_tensors=None):

        batch = super().__call__(
            features,
            return_tensors=return_tensors,
        )

        if (
            "labels" in batch
            and "decoder_input_ids" not in batch
        ):

            labels = batch["labels"].clone()

            labels[
                labels == self.label_pad_token_id
            ] = self.tokenizer.pad_token_id

            decoder_input_ids = labels.new_full(
                labels.shape,
                self.tokenizer.pad_token_id,
            )

            decoder_input_ids[:, 1:] = labels[:, :-1].clone()

            decoder_input_ids[:, 0] = (
                self.model.config.decoder_start_token_id
            )

            batch["decoder_input_ids"] = decoder_input_ids

        return batch

# Fine-tune the Model

### 7. Download and Prepare the Dataset

In [7]:
if cfg.train_mode:

    import glob
    import gdown

    os.makedirs(cfg.data_dir, exist_ok=True)

    folder_url = "https://drive.google.com/drive/folders/1Vf_TlA0eHX0LGqSu3zvZJ6jhM9cuoAjO"

    gdown.download_folder(
        folder_url,
        output=cfg.data_dir,
        quiet=False,
        use_cookies=False,
    )

    csv_files = glob.glob(
        os.path.join(cfg.data_dir, "**", cfg.train_src),
        recursive=True,
    )

    if csv_files:
        cfg.data_dir = os.path.dirname(csv_files[0])

    assert os.path.isdir(cfg.data_dir), f"{cfg.data_dir} not found"

    print(f"Dataset directory: {cfg.data_dir}")
    print(sorted(os.listdir(cfg.data_dir)))

Retrieving folder contents


Processing file 1u_XbFQzIXcCroM-GbJpZoiqgiAeHDsk2 dev_en_1000.csv
Processing file 1Ej0xIVbRxKgZOUQvgCbYm2VA7FUW4kzq dev_sa_1000.csv
Processing file 1eK5tyARSZ8IKPsnVR-rxIM3j9IQaCXIY test_en_1000.csv
Processing file 1sZ7DOxbmk4nfgljGOKfwRx7izTyyaLge test_sa_1000.csv
Processing file 1V6wj0nsrIfnwZV2iblVJoYK_e8wF5mEz train_en_10000.csv
Processing file 1i1jZ34MmF8SiDc2oHeC6kO8O_3oWITJ7 train_sa_10000.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1u_XbFQzIXcCroM-GbJpZoiqgiAeHDsk2
To: /content/data/dev_en_1000.csv
100%|██████████| 75.2k/75.2k [00:00<00:00, 63.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ej0xIVbRxKgZOUQvgCbYm2VA7FUW4kzq
To: /content/data/dev_sa_1000.csv
100%|██████████| 177k/177k [00:00<00:00, 101MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eK5tyARSZ8IKPsnVR-rxIM3j9IQaCXIY
To: /content/data/test_en_1000.csv
100%|██████████| 75.8k/75.8k [00:00<00:00, 60.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sZ7DOxbmk4nfgljGOKfwRx7izTyyaLge
To: /content/data/test_sa_1000.csv
100%|██████████| 179k/179k [00:00<00:00, 104MB/s]
Downloading...
From: https://drive.google.com/uc?id=1V6wj0nsrIfnwZV2iblVJoYK_e8wF5mEz
To: /content/data/train_en_10000.csv
100%|██████████| 783k/783k [00:00<00:00, 58.6MB/s]
Downloading...
From: https://driv

Dataset directory: data
['dev_en_1000.csv', 'dev_sa_1000.csv', 'test_en_1000.csv', 'test_sa_1000.csv', 'train_en_10000.csv', 'train_sa_10000.csv']



Download completed


### 8. Load and Preprocess the Dataset

In [8]:
if cfg.train_mode:

    _, tr_src, tr_tgt = load_pairs(cfg, cfg.train_src, cfg.train_tgt)
    _, dv_src, dv_tgt = load_pairs(cfg, cfg.dev_src, cfg.dev_tgt)

    print(f"Training samples: {len(tr_src)}")
    print(f"Validation samples: {len(dv_src)}")

    train_ds = Dataset.from_dict(
        {
            "src": tr_src,
            "tgt": tr_tgt,
        }
    )

    dev_ds = Dataset.from_dict(
        {
            "src": dv_src,
            "tgt": dv_tgt,
        }
    )

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        src_lang=cfg.src_lang,
        tgt_lang=cfg.tgt_lang,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        cfg.model_name
    ).to(DEVICE)

    model.config.use_cache = False
    model.config.decoder_start_token_id = tokenizer.convert_tokens_to_ids(
        cfg.tgt_lang
    )
    model.config.pad_token_id = tokenizer.pad_token_id

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")

    def preprocess(batch):
        return tokenizer(
            batch["src"],
            text_target=batch["tgt"],
            max_length=cfg.max_len,
            truncation=True,
        )

    train_tok = train_ds.map(
        preprocess,
        batched=True,
        remove_columns=train_ds.column_names,
    )

    dev_tok = dev_ds.map(
        preprocess,
        batched=True,
        remove_columns=dev_ds.column_names,
    )

    collator = DataCollatorWithDecoderInputs(
        tokenizer,
        model=model,
    )

    print("Dataset preprocessing completed.")

Training samples: 10000
Validation samples: 1000


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Model parameters: 615,073,792


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset preprocessing completed.


### 9. Fine-tune the Model

In [9]:
if cfg.train_mode:

    def compute_metrics(eval_preds):

        predictions, labels = eval_preds

        if isinstance(predictions, tuple):
            predictions = predictions[0]

        predictions = np.where(
            predictions != -100,
            predictions,
            tokenizer.pad_token_id,
        )

        labels = np.where(
            labels != -100,
            labels,
            tokenizer.pad_token_id,
        )

        decoded_predictions = tokenizer.batch_decode(
            predictions,
            skip_special_tokens=True,
        )

        decoded_labels = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True,
        )

        corpus_bleu, _ = compute_bleu(
            decoded_predictions,
            decoded_labels,
        )

        return {"bleu": corpus_bleu * 100}


    parameters = inspect.signature(
        Seq2SeqTrainingArguments.__init__
    ).parameters

    evaluation_argument = (
        "eval_strategy"
        if "eval_strategy" in parameters
        else "evaluation_strategy"
    )

    training_args = dict(
        output_dir=cfg.work_dir,
        num_train_epochs=cfg.epochs,
        learning_rate=cfg.lr,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        label_smoothing_factor=cfg.label_smoothing,
        predict_with_generate=True,
        generation_max_length=cfg.max_decode_len,
        generation_num_beams=cfg.beam_size,
        fp16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        report_to="none",
    )

    training_args[evaluation_argument] = "epoch"

    args = Seq2SeqTrainingArguments(**training_args)

    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=cfg.early_stop_patience
            )
        ],
    )

    trainer_parameters = inspect.signature(
        Seq2SeqTrainer.__init__
    ).parameters

    trainer_kwargs[
        "processing_class"
        if "processing_class" in trainer_parameters
        else "tokenizer"
    ] = tokenizer

    trainer = Seq2SeqTrainer(**trainer_kwargs)

    trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Bleu
1,6.698337,3.174830,13.397453
2,6.264962,3.029413,16.100504
3,5.961216,2.973655,18.188527
4,5.788948,2.945489,18.595851
5,5.655336,2.929093,19.200697
6,5.376490,2.917212,20.237277
7,5.378969,2.904273,19.731650
8,5.318149,2.891544,20.367088
9,5.281265,2.888371,20.464552
10,5.243388,2.885236,20.765328


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


### 10. Saving the Model
The fine-tuned model and tokenizer are saved for future inference and deployment.

In [10]:
if cfg.train_mode:

    trainer.save_model(cfg.model_dir)
    tokenizer.save_pretrained(cfg.model_dir)

    shutil.make_archive(
        "finetuned_model",
        "zip",
        cfg.model_dir,
    )

    print(f"Model saved to: {os.path.abspath(cfg.model_dir)}")
    print(f"Compressed file created: {os.path.abspath('finetuned_model.zip')}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/finetuned_model
Compressed file created: /content/finetuned_model.zip


### 11. Evaluating the Model and Generating Predictions

In [11]:
if cfg.train_mode:

    dev_hypotheses = translate_corpus(
        model,
        tokenizer,
        dv_src,
        cfg,
    )

    dev_corpus_bleu, dev_sentence_bleu = compute_bleu(
        dev_hypotheses,
        dv_tgt,
    )

    dev_bertscore = compute_bertscore(
        dev_hypotheses,
        dv_tgt,
        lang="en",
    )

    print("\nValidation Results")
    print("-" * 40)
    print(f"Corpus BLEU      : {dev_corpus_bleu:.4f}")
    print(f"Sentence BLEU    : {dev_sentence_bleu:.4f}")
    print(f"BERTScore (F1)   : {dev_bertscore:.4f}")

    test_ids, test_src, _ = load_pairs(
        cfg,
        cfg.test_src,
        None,
    )

    start_time = time.time()

    test_predictions = translate_corpus(
        model,
        tokenizer,
        test_src,
        cfg,
    )

    inference_time = time.time() - start_time
    total_parameters = sum(
        p.numel() for p in model.parameters()
    )

    submission = pd.DataFrame(
        {
            cfg.id_col: test_ids,
            cfg.tgt_text_col: test_predictions,
        }
    )

    submission.to_csv(
        "submission.csv",
        index=False,
        encoding="utf-8",
    )

    print("\nTest Results")
    print("-" * 40)
    print(f"Test Samples     : {len(test_predictions)}")
    print(f"Inference Time   : {inference_time:.2f} seconds")
    print(f"Model Parameters : {total_parameters:,}")

    print("\nSubmission file saved to submission.csv")

    print("\nFinal Results")
    print("-" * 40)
    print(f"Corpus BLEU      : {dev_corpus_bleu:.4f}")
    print(f"Sentence BLEU    : {dev_sentence_bleu:.4f}")
    print(f"BERTScore (F1)   : {dev_bertscore:.4f}")
    print(f"Inference Time   : {inference_time:.2f} seconds")
    print(f"Model Parameters : {total_parameters:,}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Validation Results
----------------------------------------
Corpus BLEU      : 0.2592
Sentence BLEU    : 0.2298
BERTScore (F1)   : 0.6370

Test Results
----------------------------------------
Test Samples     : 1000
Inference Time   : 90.39 seconds
Model Parameters : 615,073,792

Submission file saved to submission.csv

Final Results
----------------------------------------
Corpus BLEU      : 0.2592
Sentence BLEU    : 0.2298
BERTScore (F1)   : 0.6370
Inference Time   : 90.39 seconds
Model Parameters : 615,073,792


## 12. Inferencing
Loading the Fine-Tuned Model and Generating Predictions

**NOTE:** Change the variable private_test_path in config - to add path of new dataset for inferencing

In [12]:

import subprocess

assert os.path.isdir(
    cfg.model_dir
), f"Model directory not found: {cfg.model_dir}"

tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_dir
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    cfg.model_dir
).to(DEVICE)

model.eval()

total_parameters = sum(
    p.numel() for p in model.parameters()
)

print(f"Model parameters: {total_parameters:,}")

private_df = pd.read_csv(
    cfg.private_test_path
)

private_df[cfg.private_src_col] = (
    private_df[cfg.private_src_col]
    .fillna("")
    .astype(str)
)

private_ids = private_df[cfg.private_id_col].tolist()
private_src = private_df[cfg.private_src_col].tolist()

print(f"Test samples: {len(private_src)}")

start_time = time.time()

predictions = translate_corpus(
    model,
    tokenizer,
    private_src,
    cfg,
)

inference_time = time.time() - start_time

submission = pd.DataFrame(
    {
        cfg.id_col: private_ids,
        cfg.tgt_text_col: predictions,
    }
)

submission.to_csv(
    "submission.csv",
    index=False,
    encoding="utf-8",
)

print(f"Inference time: {inference_time:.2f} seconds")
print("Submission file saved as submission.csv")

display(submission.head())

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Model parameters: 615,073,792
Test samples: 1000
Inference time: 88.03 seconds
Submission file saved as submission.csv


,Source_id,Sentence_en
0,1,It also helps to troubleshoot errors for Eclip...
1,2,"""As it is written, We also have the spirit of ..."
2,3,Then it will automatically search for the driv...
3,4,"For all iterations, the iterator is set to the..."
4,5,"""And when he had opened the second seal, I hea..."


### 13. Download the Output Files

In [13]:
from google.colab import files

files.download("submission.csv")

if cfg.train_mode:
    files.download("finetuned_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>